In [1]:
# packages
import pandas as pd
from mod02_build_bot_predictor import train_model

### Define a function to extract predictions from the model

In [3]:
def predict_bot(df, model=None):
    """
    Predict whether each account is a bot (1) or human (0).
    """
    if model is None:
        model = train_model()

    preds = model.predict(df)
    return pd.Series(preds, index=df.index)

### Define a function to evaluate model error

In [4]:
def confusion_matrix_and_metrics(y_true, y_pred):
    """
    Computes confusion matrix and common error rates for binary classification.

    Assumes labels:
      0 = negative class
      1 = positive class

    Returns:
      dict with:
        tn, fp, fn, tp
        misclassification_rate
        false_positive_rate
        false_negative_rate
    """
    tn = fp = fn = tp = 0

    for yt, yp in zip(y_true, y_pred):
        if yt == 0 and yp == 0:
            tn += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 1 and yp == 0:
            fn += 1
        elif yt == 1 and yp == 1:
            tp += 1
        else:
            raise ValueError("Labels must be 0 or 1")

    total = tn + fp + fn + tp

    misclassification_rate = (fp + fn) / total if total > 0 else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "misclassification_rate": misclassification_rate,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
    }


### Load the data

In [5]:
TRAIN_PATH = "mod02_data/train.csv"
train = pd.read_csv(TRAIN_PATH)

TEST_PATH = "mod02_data/test.csv"
test = pd.read_csv(TEST_PATH)

### Format the data by independent vs. dependent variables

In [6]:
X_train = train.drop(columns=["is_bot"])
y_train = train['is_bot']

X_test = test.drop(columns=["is_bot"])
y_test = test['is_bot']

### Build the model on training data

In [7]:
model = train_model(X_train, y_train)

### Get the model predictions on training and test data

In [8]:
y_pred_train = predict_bot(X_train, model)
y_pred_test = predict_bot(X_test, model)

### Check results on the training set (data used to build the model)

In [9]:
confusion_matrix_and_metrics(y_train, y_pred_train)

{'tp': 113,
 'tn': 2591,
 'fp': 46,
 'fn': 250,
 'misclassification_rate': 0.09866666666666667,
 'false_positive_rate': 0.017444065225635193,
 'false_negative_rate': 0.6887052341597796}

### Check results on the test set (new data not yet seen by the model)

In [10]:
confusion_matrix_and_metrics(y_test, y_pred_test)

{'tp': 33,
 'tn': 861,
 'fp': 13,
 'fn': 93,
 'misclassification_rate': 0.106,
 'false_positive_rate': 0.014874141876430207,
 'false_negative_rate': 0.7380952380952381}

# Discussion Questions

### Based on the misclassification rate of your model, discuss your confidence in the ability to predict a bot. 

Test misclassification rate is 10.6% (106 wrong out of 1000), which looks fine until you break it down by class. False positive rate is only 1.5% (13 wrong out of 874 real humans), but false negative rate is 73.8% (93 wrong out of 126 real bots). In other words, the model catches just 33 of 126 actual bots, about 26%. On the training set the gap between train and test misclassification (9.9% vs 10.6%) is small, so the model isn't badly overfit, it's just genuinely weak at the bot class specifically. Because bots are only about 12% of the data, a model that leans toward predicting human gets rewarded with a low overall misclassification rate without actually being good at the task we care about. A more evenly distributed dataset, with a closer to 50/50 split of bots and humans, would likely help the model actually learn what separates the two classes instead of just defaulting to the majority label. So overall I'd say I'm not confident in this model's ability to reliably flag bots, even though misclassification rate looks decent

### What are potential ramifications of false positives from the model?

13 of 874 real humans (1.5%) got flagged as bots on the test set. Each of those is a real person who could face consequences depending on how the flag is used, things like being rate-limited, temporarily locked out, shadow-banned, or having their posts removed or deprioritized, all for behavior that was actually legitimate. A 1.5% rate sounds small, but at real platform scale (millions of users) that turns into a large number of wrongly penalized accounts. 

### What are potential ramifications of false negatives from the model?

93 of 126 actual bots (73.8%) went undetected on the test set, meaning only 33 were caught. That's the more serious failure mode here since the whole point of the model is catching bots. Undetected bots at that rate could keep operating freely: spreading misinformation, inflating engagement metrics like likes, follows, or trending topics, scraping data, or running spam and scam campaigns, all while looking like normal accounts to this model. Given that roughly 3 out of 4 bots slip through, I wouldn't trust this model as a standalone gate for bot detection. It would need to be paired with other signals or used as one input into a larger detection system rather than the final decision-maker.